# Контролируемые GRU priors: P0, P0.1, P1 и P5

Здесь проверяется, влияет ли доля main-group структур в pretraining-корпусе на то, насколько хорошо модель понимает FLP-like SMILES до дообучения на FLP.

Во всех опытах одинаковы архитектура GRU, tokenizer, размер корпуса, число эпох и параметры обучения. Меняется только доля main-group структур:

| prior | main-group доля | число структур |
|---|---:|---:|
| P0 | 0% | 0 |
| P0.1 | 0.1% | 150 |
| P1 | 1% | 1500 |
| P5 | 5% | 7500 |

Каждый prior обучается с тремя random seed. Checkpoint выбирается по BPC на общем validation-наборе, который не содержит main-group элементов. После обучения считаются BPC на обычных, main-group и FLP структурах, а также сохраняются zero-shot генерации.

Меньший BPC означает, что модель лучше предсказывает соответствующий тип SMILES.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("tokenizers") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "tokenizers>=0.20",
    ])

print("Среда готова")

In [ ]:
import base64
import json
import math
import random
import shutil
from pathlib import Path
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from tokenizers import Tokenizer


WORK_DIR = Path.cwd()
EXTRACT_DIR = WORK_DIR / "controlled_prior_data"
RESULT_DIR = WORK_DIR / "results" / "controlled_prior_pretraining"
WEIGHT_DIR = WORK_DIR / "models" / "controlled_prior_pretraining"

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
WEIGHT_DIR.mkdir(parents=True, exist_ok=True)

PRIORS = ["P0", "P0.1", "P1", "P5"]
TRAINING_SEEDS = [11, 22, 33]
GENERATION_SEEDS = [101, 202, 303]

EPOCHS = 3
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
MAX_LENGTH = 202
SAMPLES_PER_GENERATION_SEED = 500
TEMPERATURE = 1.0

EMBEDDING_SIZE = 128
HIDDEN_SIZE = 256
NUM_LAYERS = 2
DROPOUT = 0.2

TOKENIZER_ID = "ibm-research/MoLFormer-XL-both-10pct"
TOKENIZER_REVISION = "abb341ed7cee8c7783088d70bf9f86dff266844e"

COLORS = {
    "green": "#356859",
    "violet": "#7663A5",
    "wine": "#963F52",
    "olive": "#7D8748",
    "gray": "#686D70",
}

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.2,
    "legend.frameon": False,
})

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 1. Данные

Перед запуском нужно добавить в Colab или Kaggle файл `controlled_prior_corpora.zip`. Распаковывать его вручную не нужно.

In [ ]:
manifest_candidates = (
    list(Path("/kaggle/input").rglob("corpus_manifest.json"))
    + list(Path(".").rglob("corpus_manifest.json"))
)

if manifest_candidates:
    INPUT_DIR = manifest_candidates[0].parent
else:
    archive_candidates = (
        list(Path("/content").glob("controlled_prior_corpora.zip"))
        + list(Path(".").glob("controlled_prior_corpora.zip"))
    )
    if not archive_candidates:
        raise FileNotFoundError(
            "Не найден controlled_prior_corpora.zip"
        )

    archive_path = archive_candidates[0]
    print("Архив:", archive_path)
    with ZipFile(archive_path) as archive:
        archive.extractall(EXTRACT_DIR)
    INPUT_DIR = EXTRACT_DIR


def input_file(*names):
    for name in names:
        matches = [
            path
            for path in INPUT_DIR.rglob(name)
            if path.is_file()
        ]
        if matches:
            return matches[0]

    for name in names:
        directories = [
            path
            for path in INPUT_DIR.rglob(name)
            if path.is_dir()
        ]
        for directory in directories:
            files = [
                path
                for path in directory.rglob("*")
                if path.is_file()
            ]
            if files:
                return max(files, key=lambda path: path.stat().st_size)

    raise FileNotFoundError("Не найден файл: " + " или ".join(names))


def table_file(name):
    return input_file(f"{name}.csv.gz", f"{name}.csv")


print("Данные:", INPUT_DIR)
summary = pd.read_csv(table_file("corpus_summary"))
balance = pd.read_csv(table_file("matching_balance"))
leakage = json.loads(
    input_file("leakage_report.json").read_text(encoding="utf-8")
)
corpus_manifest = json.loads(
    input_file("corpus_manifest.json").read_text(encoding="utf-8")
)

print(summary.round(4).to_string(index=False))
print()
print("Проверка утечек:")
print(pd.Series(leakage).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

axes[0].bar(
    summary["prior"],
    summary["main_group_fraction"] * 100,
    color=[
        COLORS["gray"],
        COLORS["green"],
        COLORS["violet"],
        COLORS["wine"],
    ],
)
axes[0].set_title("Доля main-group структур")
axes[0].set_ylabel("% корпуса")

for prior, color in zip(
    ["P0.1", "P1", "P5"],
    [COLORS["green"], COLORS["violet"], COLORS["wine"]],
):
    part = balance[balance["prior"] == prior]
    axes[1].plot(
        part["descriptor"],
        part["standardized_mean_difference"].abs(),
        marker="o",
        label=prior,
        color=color,
    )

axes[1].axhline(0.2, color=COLORS["gray"], linestyle="--", linewidth=1)
axes[1].set_title("Баланс заменённых структур")
axes[1].set_ylabel("|standardized mean difference|")
axes[1].tick_params(axis="x", rotation=35)
axes[1].legend()

fig.tight_layout()
fig.savefig(RESULT_DIR / "corpus_design.png", bbox_inches="tight")
plt.show()

Все корпуса содержат по 150 000 уникальных SMILES. Значения `|SMD|` ниже 0.2 означают, что main-group структуры и заменённые ими обычные молекулы близки по размеру и простым дескрипторам.

## 2. Tokenizer и модель

In [ ]:
TOKENIZER_B64 = """..."""
tokenizer_path = WORK_DIR / "molformer_tokenizer.json"
tokenizer_path.write_bytes(base64.b64decode(TOKENIZER_B64))

tokenizer = Tokenizer.from_file(str(tokenizer_path))
VOCAB_SIZE = tokenizer.get_vocab_size()
PAD_TOKEN_ID = tokenizer.token_to_id("<pad>")
BOS_TOKEN_ID = tokenizer.token_to_id("<bos>")
EOS_TOKEN_ID = tokenizer.token_to_id("<eos>")

print("Размер словаря:", VOCAB_SIZE)
print("BOS/EOS/PAD:", BOS_TOKEN_ID, EOS_TOKEN_ID, PAD_TOKEN_ID)

In [ ]:
class SmilesDataset(Dataset):
    def __init__(self, smiles):
        self.sequences = []
        self.character_counts = []

        for value in smiles:
            tokens = tokenizer.encode(
                value,
                add_special_tokens=False,
            ).ids
            tokens = [BOS_TOKEN_ID] + tokens + [EOS_TOKEN_ID]
            self.sequences.append(torch.tensor(tokens, dtype=torch.long))
            self.character_counts.append(len(value))

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, index):
        return self.sequences[index], self.character_counts[index]


def make_batch(items):
    sequences, character_counts = zip(*items)
    inputs = pad_sequence(
        [sequence[:-1] for sequence in sequences],
        batch_first=True,
        padding_value=PAD_TOKEN_ID,
    )
    targets = pad_sequence(
        [sequence[1:] for sequence in sequences],
        batch_first=True,
        padding_value=PAD_TOKEN_ID,
    )
    return inputs, targets, sum(character_counts)


class SmilesGRU(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(
            VOCAB_SIZE,
            EMBEDDING_SIZE,
            padding_idx=PAD_TOKEN_ID,
        )
        self.gru = nn.GRU(
            EMBEDDING_SIZE,
            HIDDEN_SIZE,
            num_layers=NUM_LAYERS,
            dropout=DROPOUT,
            batch_first=True,
        )
        self.output = nn.Linear(HIDDEN_SIZE, VOCAB_SIZE)

    def forward(self, tokens, hidden=None):
        embedded = self.embedding(tokens)
        values, hidden = self.gru(embedded, hidden)
        return self.output(values), hidden


parameter_count = sum(
    parameter.numel()
    for parameter in SmilesGRU().parameters()
)
print("Параметров в GRU:", f"{parameter_count:,}")

Архитектура совпадает с GRU baseline: embedding 128, два GRU-слоя по 256 скрытых признаков и dropout 0.2.

## 3. Общие validation-наборы

In [ ]:
general_smiles = pd.read_csv(
    table_file("validation_general")
)["smiles"].astype(str).tolist()

main_group_smiles = pd.read_csv(
    table_file("validation_main_group")
)["smiles"].astype(str).tolist()

flp_smiles = pd.read_csv(
    table_file("validation_flp")
)["smiles"].astype(str).tolist()

general_loader = DataLoader(
    SmilesDataset(general_smiles),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=make_batch,
)
main_group_loader = DataLoader(
    SmilesDataset(main_group_smiles),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=make_batch,
)
flp_loader = DataLoader(
    SmilesDataset(flp_smiles),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=make_batch,
)

print("Обычные структуры:", len(general_smiles))
print("Main-group структуры:", len(main_group_smiles))
print("FLP validation:", len(flp_smiles))

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def calculate_bpc(model, loader):
    model.eval()
    total_loss = 0.0
    total_characters = 0
    loss_function = nn.CrossEntropyLoss(
        ignore_index=PAD_TOKEN_ID,
        reduction="sum",
    )

    with torch.no_grad():
        for inputs, targets, character_count in loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            logits, _ = model(inputs)
            total_loss += loss_function(
                logits.reshape(-1, VOCAB_SIZE),
                targets.reshape(-1),
            ).item()
            total_characters += character_count

    return total_loss / (total_characters * math.log(2))


def train_prior(prior, training_seed, train_dataset):
    set_seed(training_seed)
    model = SmilesGRU().to(device)

    generator = torch.Generator().manual_seed(training_seed)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        collate_fn=make_batch,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=1e-4,
    )
    loss_function = nn.CrossEntropyLoss(
        ignore_index=PAD_TOKEN_ID,
    )

    best_general_bpc = float("inf")
    best_epoch = 0
    best_weights = None
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_losses = []

        for inputs, targets, _ in train_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()
            logits, _ = model(inputs)
            loss = loss_function(
                logits.reshape(-1, VOCAB_SIZE),
                targets.reshape(-1),
            )
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item())

        general_bpc = calculate_bpc(model, general_loader)
        main_group_bpc = calculate_bpc(model, main_group_loader)
        flp_bpc = calculate_bpc(model, flp_loader)

        history.append({
            "prior": prior,
            "training_seed": training_seed,
            "epoch": epoch,
            "train_loss": np.mean(train_losses),
            "general_bpc": general_bpc,
            "main_group_bpc": main_group_bpc,
            "flp_bpc": flp_bpc,
        })

        print(
            prior,
            "| seed", training_seed,
            "| epoch", epoch,
            "| loss", round(np.mean(train_losses), 3),
            "| BPC", round(general_bpc, 3),
            round(main_group_bpc, 3),
            round(flp_bpc, 3),
        )

        if general_bpc < best_general_bpc:
            best_general_bpc = general_bpc
            best_epoch = epoch
            best_weights = {
                name: value.detach().cpu().clone()
                for name, value in model.state_dict().items()
            }

    model.load_state_dict(best_weights)
    model.eval()
    return model, pd.DataFrame(history), best_epoch

Checkpoint выбирается только по обычному validation-набору. Main-group и FLP BPC не участвуют в выборе и поэтому остаются независимыми показателями совместимости prior с целевым доменом.

## 4. Zero-shot генерация

In [ ]:
def generate_smiles(model, generation_seed):
    generator = torch.Generator(device=device).manual_seed(generation_seed)
    generated = []
    reached_limit = []

    while len(generated) < SAMPLES_PER_GENERATION_SEED:
        batch_size = min(
            128,
            SAMPLES_PER_GENERATION_SEED - len(generated),
        )
        current = torch.full(
            (batch_size, 1),
            BOS_TOKEN_ID,
            dtype=torch.long,
            device=device,
        )
        sequences = current
        hidden = None
        finished = torch.zeros(
            batch_size,
            dtype=torch.bool,
            device=device,
        )

        for _ in range(MAX_LENGTH - 1):
            logits, hidden = model(current, hidden)
            probabilities = torch.softmax(
                logits[:, -1, :] / TEMPERATURE,
                dim=-1,
            )
            next_token = torch.multinomial(
                probabilities,
                num_samples=1,
                generator=generator,
            )
            next_token[finished] = PAD_TOKEN_ID
            sequences = torch.cat([sequences, next_token], dim=1)
            finished |= next_token[:, 0].eq(EOS_TOKEN_ID)
            current = next_token

            if finished.all():
                break

        decoded = tokenizer.decode_batch(
            sequences[:, 1:].cpu().tolist(),
            skip_special_tokens=True,
        )
        generated.extend(decoded)
        reached_limit.extend((~finished).cpu().tolist())

    return generated, reached_limit

## 5. Обучение

In [ ]:
all_histories = []
run_rows = []

for prior in PRIORS:
    print()
    print("Подготовка корпуса", prior)
    train_smiles = pd.read_csv(
        table_file(prior),
        usecols=["smiles"],
    )["smiles"].astype(str).tolist()
    train_dataset = SmilesDataset(train_smiles)

    for training_seed in TRAINING_SEEDS:
        model, history, best_epoch = train_prior(
            prior,
            training_seed,
            train_dataset,
        )
        all_histories.append(history)

        scores = {
            "prior": prior,
            "training_seed": training_seed,
            "best_epoch": best_epoch,
            "general_bpc": calculate_bpc(model, general_loader),
            "main_group_bpc": calculate_bpc(model, main_group_loader),
            "flp_bpc": calculate_bpc(model, flp_loader),
        }
        scores["main_group_gap"] = (
            scores["main_group_bpc"] - scores["general_bpc"]
        )
        scores["flp_gap"] = scores["flp_bpc"] - scores["general_bpc"]
        run_rows.append(scores)

        torch.save(
            {
                "state_dict": model.state_dict(),
                "prior": prior,
                "training_seed": training_seed,
                "best_epoch": best_epoch,
                "vocab_size": VOCAB_SIZE,
                "embedding_size": EMBEDDING_SIZE,
                "hidden_size": HIDDEN_SIZE,
                "num_layers": NUM_LAYERS,
                "dropout": DROPOUT,
                "max_length": MAX_LENGTH,
                "tokenizer_id": TOKENIZER_ID,
                "tokenizer_revision": TOKENIZER_REVISION,
            },
            WEIGHT_DIR / f"gru_{prior}_seed_{training_seed}.pt",
        )

        for generation_seed in GENERATION_SEEDS:
            smiles, limits = generate_smiles(model, generation_seed)
            pd.DataFrame({
                "generated_smiles": smiles,
                "reached_max_length": limits,
                "prior": prior,
                "training_seed": training_seed,
                "generation_seed": generation_seed,
            }).to_csv(
                RESULT_DIR
                / (
                    f"zero_shot_{prior}_train_{training_seed}"
                    f"_generation_{generation_seed}.csv"
                ),
                index=False,
            )

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    del train_dataset, train_smiles

history_table = pd.concat(all_histories, ignore_index=True)
run_table = pd.DataFrame(run_rows)

history_table.to_csv(RESULT_DIR / "training_history.csv", index=False)
run_table.to_csv(RESULT_DIR / "frozen_bpc_runs.csv", index=False)

print()
print(run_table.round(4).to_string(index=False))

## 6. Сводка frozen BPC

In [ ]:
metric_columns = [
    "general_bpc",
    "main_group_bpc",
    "flp_bpc",
    "main_group_gap",
    "flp_gap",
]

frozen_summary = (
    run_table.groupby("prior")[metric_columns]
    .agg(["mean", "std"])
    .round(4)
)
frozen_summary.to_csv(RESULT_DIR / "frozen_bpc_summary.csv")
print(frozen_summary.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
prior_colors = {
    "P0": COLORS["gray"],
    "P0.1": COLORS["green"],
    "P1": COLORS["violet"],
    "P5": COLORS["wine"],
}

means = run_table.groupby("prior")[[
    "general_bpc",
    "main_group_bpc",
    "flp_bpc",
]].mean().reindex(PRIORS)

x = np.arange(len(PRIORS))
width = 0.24
for offset, metric, label, color in [
    (-width, "general_bpc", "обычные", COLORS["gray"]),
    (0, "main_group_bpc", "main-group", COLORS["olive"]),
    (width, "flp_bpc", "FLP", COLORS["wine"]),
]:
    axes[0].bar(
        x + offset,
        means[metric],
        width,
        label=label,
        color=color,
        alpha=0.9,
    )

axes[0].set_xticks(x, PRIORS)
axes[0].set_ylabel("bits per character")
axes[0].set_title("Frozen BPC")
axes[0].legend()

for prior in PRIORS:
    part = run_table[run_table["prior"] == prior]
    axes[1].scatter(
        [prior] * len(part),
        part["flp_gap"],
        s=55,
        color=prior_colors[prior],
        alpha=0.8,
    )
    axes[1].plot(
        [prior],
        [part["flp_gap"].mean()],
        marker="_",
        markersize=20,
        color="black",
    )

axes[1].set_ylabel("FLP BPC - general BPC")
axes[1].set_title("Разрыв между FLP и обычным языком")

fig.tight_layout()
fig.savefig(RESULT_DIR / "frozen_bpc_comparison.png", bbox_inches="tight")
plt.show()

На первом графике сравнивается абсолютное качество языка. На втором убрана общая разница между удачными и неудачными обучениями: меньший `FLP gap` означает, что prior относительно лучше переносится на FLP-like SMILES.

## 7. Сохранение результатов

In [ ]:
experiment_config = {
    "priors": PRIORS,
    "training_seeds": TRAINING_SEEDS,
    "generation_seeds": GENERATION_SEEDS,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "max_length": MAX_LENGTH,
    "samples_per_generation_seed": SAMPLES_PER_GENERATION_SEED,
    "temperature": TEMPERATURE,
    "embedding_size": EMBEDDING_SIZE,
    "hidden_size": HIDDEN_SIZE,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
    "tokenizer_id": TOKENIZER_ID,
    "tokenizer_revision": TOKENIZER_REVISION,
    "corpus_manifest": corpus_manifest,
}

(RESULT_DIR / "experiment_config.json").write_text(
    json.dumps(experiment_config, indent=2),
    encoding="utf-8",
)

results_zip = shutil.make_archive(
    str(WORK_DIR / "controlled_prior_pretraining_results"),
    "zip",
    RESULT_DIR,
)
weights_zip = shutil.make_archive(
    str(WORK_DIR / "controlled_prior_pretraining_weights"),
    "zip",
    WEIGHT_DIR,
)

print("Результаты:", results_zip)
print("Веса:", weights_zip)

Скачайте два ZIP-файла из панели файлов:

- `controlled_prior_pretraining_results.zip`;
- `controlled_prior_pretraining_weights.zip`.

Первый нужен для анализа, второй — для следующего этапа с FLP fine-tuning.